# ML Pipeline Demo

This notebook demonstrates the **full demo flow** of the project:

1. Download the last 6 months of hourly market data from Binance
2. Preprocess ETH, BTC, and ETHBTC into a single feature dataframe
3. Load the training and test configs
4. Prepare a notebook-friendly runtime that disables model persistence
5. Run the grid-search pipeline
6. Inspect selected models and example predictions

> **Note:** the demo version disables model saving, log files, and Excel outputs.  

## 0. Optional setup for Google Colab

If you run this notebook **inside the repository**, you can skip the next cell.

If you open the notebook directly in **Colab from GitHub**, set `REPO_URL` and run the setup cell once so the rest of the project files become available in the runtime.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ixor911/ML_Pipeline_Demo.git"

if IS_COLAB and not Path("core").exists():
    if not REPO_URL:
        raise ValueError("Set REPO_URL before running this cell in Colab.")
    subprocess.run(["git", "clone", REPO_URL, "repo"], check=True)
    os.chdir("repo")
    print("Repository cloned into Colab runtime.")
else:
    print("Repository files already available. Skipping clone step.")

## 1. Download raw market data

We fetch the last 6 months of **ETHUSDT**, **BTCUSDT**, and **ETHBTC** hourly candles directly from Binance.

In [17]:
import pandas as pd
from IPython.display import display

from core.features.DataProvider import DataProvider

end = pd.Timestamp.utcnow().floor("h")
start = end - pd.DateOffset(months=6)

print(f"Downloading hourly candles from {start} to {end}...")

eth_df = DataProvider.download_klines_months(symbol="ETHUSDT", months=6)
btc_df = DataProvider.download_klines_months(symbol="BTCUSDT", months=6)
ethbtc_df = DataProvider.download_klines_months(symbol="ETHBTC", months=6)

print(f"ETH rows    : {len(eth_df)}")
print(f"BTC rows    : {len(btc_df)}")
print(f"ETHBTC rows : {len(ethbtc_df)}")

display(eth_df.head())

ETH rows    : 4367
BTC rows    : 4367
ETHBTC rows : 4367


,open_time,open,high,low,close,volume,close_time,quote,trades,taker_buy_base,taker_buy_quote
0,2025-10-23 19:00:00+00:00,3870.14,3871.36,3845.00,3859.40,16742.6248,2025-10-23 19:59:59.999000+00:00,6.456791e+07,150523,7452.4666,2.874262e+07
1,2025-10-23 20:00:00+00:00,3859.40,3860.10,3818.72,3829.20,13589.4972,2025-10-23 20:59:59.999000+00:00,5.220621e+07,150297,5376.6225,2.065362e+07
2,2025-10-23 21:00:00+00:00,3829.20,3837.18,3821.50,3828.91,6801.2145,2025-10-23 21:59:59.999000+00:00,2.603998e+07,96263,3195.3835,1.223366e+07
3,2025-10-23 22:00:00+00:00,3828.90,3857.02,3826.91,3850.96,8196.1043,2025-10-23 22:59:59.999000+00:00,3.151067e+07,94017,4597.4002,1.767708e+07
4,2025-10-23 23:00:00+00:00,3850.96,3860.00,3845.28,3856.80,9763.8476,2025-10-23 23:59:59.999000+00:00,3.762412e+07,89152,2974.3666,1.146272e+07


## 2. Preprocess the raw candles

The preprocessing step combines ETH, BTC, and ETHBTC into one processed dataset with:
- ETH technical and candle features
- BTC context features
- ETHBTC relative-strength features
- cross-market features

In [18]:
from core.features.preprocessor.Preprocessor import Preprocessor

processed_df = Preprocessor.preprocess_and_save(
    eth_df=eth_df,
    btc_df=btc_df,
    ethbtc_df=ethbtc_df,
    months=6
)

print(f"Processed rows    : {len(processed_df)}")
print(f"Processed columns : {len(processed_df.columns)}")
display(processed_df.head())

Processed rows    : 4308
Processed columns : 86


,open_time_eth,open_eth,high_eth,low_eth,close_eth,volume_eth,close_time,quote_eth,trades_eth,taker_buy_base_eth,taker_buy_quote_eth,sma20,sma50,ema12,ema26,macd_line,signal_line,macd_hist,rsi14,atr14_proxy_pct,bb_width_pct,ret_1h_pct,ret_3h_pct,ret_6h_pct,ret_24h_pct,ret_acceleration,vol_spike,rvol20,taker_buy_ratio,taker_delta_norm,quote_per_trade,clv,body_ratio,upper_wick_ratio,lower_wick_ratio,wick_imbalance,range_pct,range_compression,pctB,trend_sign,is_high_vol,regime,vol_ret_1h_pct,vol_zscore_24,cmf20,mfi14,vol_vola_coupling,rvol_anom,rvol_anom_signed,ethbtc_open_time,btc_atr14_proxy_pct,btc_bb_width_pct,btc_ema12,btc_ema26,btc_macd_hist,btc_macd_line,btc_ret_1h_lag1,btc_ret_1h_lag2,btc_ret_1h_lag3,btc_ret_1h_pct,btc_ret_24h_pct,btc_rsi14,btc_shock_24h,btc_signal_line,btc_sma20,btc_sma50,close_btc,ethbtc_ema12,ethbtc_ema26,ethbtc_macd_hist,ethbtc_macd_line,ethbtc_ret_1h_pct,ethbtc_ret_24h_pct,ethbtc_rsi14,ethbtc_signal_line,high_btc,low_btc,open_btc,quote_btc,taker_buy_base_btc,taker_buy_quote_btc,trades_btc,volume_btc,eth_ret_1h_pct,rs_1h_pct,rs_24h_pct
0,2025-10-26 06:00:00+00:00,3942.85,3946.41,3938.66,3944.88,3419.1602,2025-10-26 06:59:59.999000+00:00,1.348336e+07,34179.0,1559.9093,6.151379e+06,3943.9455,3938.6004,3942.316193,3939.929753,2.386439,4.060904,-1.674465,50.164420,0.426538,0.492842,0.051486,0.332162,-0.136446,0.410819,-0.022627,False,0.630211,0.456220,-0.087560,394.492594,0.802581,0.261935,0.197419,0.540645,-0.343226,0.196457,0.486009,0.524039,0,0,range,NaN,NaN,NaN,NaN,0.002688,-0.369789,NaN,2025-10-26 06:00:00+00:00,0.215985,0.269997,111534.552250,111445.348271,-28.833104,89.203979,0.165951,0.152741,-0.169037,0.018483,0.242021,52.379543,False,118.037083,111556.9900,111239.4228,111635.95,0.035347,0.035353,-0.000006,-0.000006,0.028305,0.170068,47.875364,-1.139026e-07,111686.65,111548.75,111615.33,1.743419e+07,57.01429,6.363046e+06,31850.0,156.20448,NaN,NaN,0.168798
1,2025-10-26 07:00:00+00:00,3944.88,3956.59,3941.71,3951.57,5105.0991,2025-10-26 07:59:59.999000+00:00,2.015987e+07,36996.0,2863.6659,1.130870e+07,3944.5085,3938.1478,3943.739855,3940.791994,2.947861,3.838296,-0.890435,53.346212,0.436915,0.497777,0.169587,0.295437,0.468582,0.576494,0.118101,False,0.916825,0.560951,0.121902,544.920153,0.662634,0.449597,0.337366,0.213038,0.124328,0.376559,0.942092,0.679820,1,0,trend_up,49.308567,NaN,NaN,NaN,0.004006,-0.083175,-0.083175,2025-10-26 07:00:00+00:00,0.212860,0.277768,111563.694981,111465.987659,-16.263809,97.707322,0.018483,0.165951,0.152741,0.078855,0.210226,54.528721,False,113.971131,111564.9985,111248.6752,111723.98,0.035352,0.035355,-0.000002,-0.000003,0.113186,0.425773,51.240854,-6.759664e-07,111724.96,111620.02,111635.96,1.898354e+07,73.31342,8.187782e+06,33958.0,169.97811,0.169587,0.090732,0.366268
2,2025-10-26 08:00:00+00:00,3951.57,3967.84,3951.08,3960.05,6354.7651,2025-10-26 08:59:59.999000+00:00,2.514900e+07,57787.0,2979.9169,1.179342e+07,3944.8425,3937.7000,3946.249108,3942.218513,4.030595,3.876756,0.153840,57.086307,0.394205,0.518077,0.214598,0.436233,0.750531,0.155289,0.045011,False,1.128895,0.468942,-0.062116,435.201758,0.535203,0.505967,0.464797,0.029236,0.435561,0.423227,1.068930,0.872052,0,0,range,24.478780,NaN,NaN,NaN,0.004450,0.128895,0.128895,2025-10-26 08:00:00+00:00,0.194328,0.257550,111596.501907,111489.021165,-5.192311,107.480741,0.078855,0.018483,0.165951,0.047403,0.011757,55.820539,False,112.673053,111558.8940,111254.9268,111776.94,0.035363,0.035360,0.000003,0.000003,0.113058,0.113058,54.410801,-3.389835e-09,111850.00,111723.98,111723.98,2.687740e+07,121.39430,1.357170e+07,41510.0,240.40616,0.214598,0.167196,0.143532
3,2025-10-26 09:00:00+00:00,3960.05,4011.25,3956.41,3987.91,28363.8612,2025-10-26 09:59:59.999000+00:00,1.131103e+08,234200.0,14363.2978,5.728586e+07,3947.0960,3938.4738,3952.658476,3945.603068,7.055409,4.512486,2.542922,66.568716,0.467466,0.701892,0.703526,1.090781,1.426566,1.152064,0.488928,True,4.199851,0.506460,0.012920,482.964735,0.574398,0.508023,0.425602,0

## 3. Load training and test configs

Here we load:

- `model_train_basic_grid` — the training grid
- `model_test_basic` — the test slicing config

In [19]:
from pprint import pprint

from core.io.loader import load_config, load_config_grid

training_configs = list(load_config_grid("model_train_basic_grid"))
test_config = load_config("model_test_basic")

print(f"Training configs loaded: {len(training_configs)}")
print("\nExample training config:")
pprint(training_configs[0])

print("\nTest config:")
pprint(test_config)

Training configs loaded: 40

Example training config:
{'builder': {'extra_drop': None,
             'feature_groups': ['eth_trend_core', 'eth_macd_pack'],
             'features_include': None,
             'future_ret_col': 'future_ret',
             'horizon': 1,
             'regime_filter': ['trend_up'],
             'tau_pct': [0.03, None]},
 'datapath': 'ETHUSDT_6_1h.csv',
 'model': {'batch_size': 64,
           'depth': 3,
           'dropout': 0.2,
           'epochs': 25,
           'hidden': 64,
           'lr': 0.001,
           'patience': 6,
           'seed': 42,
           'task': 'classification',
           'val_share': 0.2,
           'verbose': False,
           'weight_decay': 0.0001},
 'slicing': {'candles': [3000, 1000],
             'end_date': None,
             'from_tail': True,
             'time_col': 'open_time_eth',
             'type': 'candles'}}

Test config:
{'datapath': 'ETHUSDT_6_1h.csv',
 'slicing': {'candles': [1000],
             'end_date': None,

## 4. Build test candles from the processed dataframe

The grid-search pipeline evaluates all candidates on one shared test candle window.
We build that window from the freshly processed dataset using the test slicing config.


In [20]:
from core.target.slicing import slice_data

test_windows = slice_data(
    df=processed_df,
    slicing_config=test_config["slicing"],
)
test_candles = test_windows[0]

print(f"Test candles rows: {len(test_candles)}")
display(test_candles.head())


Test candles rows: 1000


,open_time_eth,open_eth,high_eth,low_eth,close_eth,volume_eth,close_time,quote_eth,trades_eth,taker_buy_base_eth,taker_buy_quote_eth,sma20,sma50,ema12,ema26,macd_line,signal_line,macd_hist,rsi14,atr14_proxy_pct,bb_width_pct,ret_1h_pct,ret_3h_pct,ret_6h_pct,ret_24h_pct,ret_acceleration,vol_spike,rvol20,taker_buy_ratio,taker_delta_norm,quote_per_trade,clv,body_ratio,upper_wick_ratio,lower_wick_ratio,wick_imbalance,range_pct,range_compression,pctB,trend_sign,is_high_vol,regime,vol_ret_1h_pct,vol_zscore_24,cmf20,mfi14,vol_vola_coupling,rvol_anom,rvol_anom_signed,ethbtc_open_time,btc_atr14_proxy_pct,btc_bb_width_pct,btc_ema12,btc_ema26,btc_macd_hist,btc_macd_line,btc_ret_1h_lag1,btc_ret_1h_lag2,btc_ret_1h_lag3,btc_ret_1h_pct,btc_ret_24h_pct,btc_rsi14,btc_shock_24h,btc_signal_line,btc_sma20,btc_sma50,close_btc,ethbtc_ema12,ethbtc_ema26,ethbtc_macd_hist,ethbtc_macd_line,ethbtc_ret_1h_pct,ethbtc_ret_24h_pct,ethbtc_rsi14,ethbtc_signal_line,high_btc,low_btc,open_btc,quote_btc,taker_buy_base_btc,taker_buy_quote_btc,trades_btc,volume_btc,eth_ret_1h_pct,rs_1h_pct,rs_24h_pct
0,2026-03-13 02:00:00+00:00,2124.75,2127.12,2102.42,2105.81,13893.8920,2026-03-13 02:59:59.999000+00:00,2.937212e+07,186281.0,5631.9422,1.190692e+07,2069.2700,2052.1516,2083.542265,2069.216918,14.325347,8.608894,5.716452,62.403491,1.365576,2.126821,-0.891865,1.557255,2.096414,4.007567,-1.154702,False,0.618929,0.405382,-0.189237,157.676405,0.137247,0.766802,0.095951,0.137247,-0.041296,1.172945,0.972251,0.915136,1,1,trend_up,-11.738042,-0.382992,0.016872,63.563167,0.008452,-0.381071,0.381071,2026-03-13 02:00:00+00:00,1.009802,1.478813,70684.485527,70393.142055,141.500888,291.343471,0.275311,1.324429,0.521121,-0.737509,2.680520,59.190445,False,149.842584,70355.1005,70114.9464,71143.80,0.029479,0.029397,0.000022,0.000082,-0.168634,1.300479,62.569134,0.000060,71696.68,71045.52,71672.38,7.105742e+07,444.89169,3.177939e+07,147297.0,994.81538,-0.891865,-0.154357,1.327047
1,2026-03-13 03:00:00+00:00,2105.81,2123.58,2102.20,2118.85,11965.2095,2026-03-13 03:59:59.999000+00:00,2.525016e+07,133309.0,5766.3768,1.217520e+07,2073.1775,2053.8066,2088.974224,2072.893443,16.080781,10.103272,5.977509,65.407057,1.332967,2.264568,0.619239,-0.016044,2.726642,4.718343,1.511105,False,0.535717,0.482183,-0.035634,189.410786,0.778765,0.609916,0.221235,0.168849,0.052385,1.009038,0.833535,0.986410,1,1,trend_up,-13.881514,-0.489652,0.006020,61.914629,0.007141,-0.464283,-0.464283,2026-03-13 03:00:00+00:00,0.971565,1.569693,70795.449292,70468.150051,141.965326,327.299241,-0.737509,0.275311,1.324429,0.368198,2.821074,61.612944,False,185.333915,70434.0785,70141.6362,71405.75,0.029510,0.029418,0.000025,0.000092,0.270270,1.853123,65.873317,0.000067,71508.87,70964.43,71143.80,6.605695e+07,517.00627,3.681469e+07,125941.0,927.82134,0.619239,0.251041,1.897269
2,2026-03-13 04:00:00+00:00,2118.86,2120.06,2109.63,2111.68,5461.6899,2026-03-13 04:59:59.999000+00:00,1.154701e+07,89249.0,2411.5234,5.099128e+06,2076.3715,2055.5060,2092.467420,2075.766521,16.700899,11.422797,5.278102,62.452673,1.179257,2.325252,-0.338391,-0.615599,2.245679,4.378945,-0.957630,False,0.248882,0.441597,-0.116805,129.379659,0.196548,0.688399,0.115053,0.196548,-0.081496,0.493920,0.416751,0.865657,1,0,trend_up,-54.353579,-0.874033,-0.011213,70.362353,0.002935,-0.751118,0.751118,2026-03-13 04:00:00+00:00,0.869117,1.636511,70887.524785,70536.727084,132.371029,350.797701,0.368198,-0.737509,0.275311,-0.016539,2.914634,61.435876,False,218.426672,70507.9070,70173.3006,71393.94,0.029521,0.029430,0.000019,0.000091,-0.336927,1.405554,58.876927,0.000072,71493.77,71214.01,71405.76,5.570447e+07,429.05759,3.061340e+07,105161.0,780.92498,-0.338391,-0.321852,1.464311
3,2026-03-13 05:00:00+00:00,2111.67,2115.24,2103.54,2110.47,7222.6564,2026-03-13 05:59:59.999000+00:00,1.523829e+07,102715.0,2961.3351,6.248466e+06,2079.1125,2057.2920,2095.237048,2078.337149,16.899899,12.518218,4.381681,61.944170,1.074168,2.379562,-0.057300,0.221293,1.781994,4.125614,0.281091,

## 5. Run the grid-search pipeline

This step:
- trains models for all training configs combinations
- expands each trained model into category-specific candidates
- evaluates them on the shared test candles
- keeps only the strongest candidates in `models_ram`


In [21]:
import os

from core.pipelines.grid_search_pipeline import (
    grid_search_pipeline,
    print_models_ram_metrics,
)

LIMIT = 3 # limit models per category

models_ram = {}

models_ram = grid_search_pipeline(
    training_configs=training_configs,
    test_candles=test_candles,
    models_ram=models_ram,
    limit=LIMIT
)

print("Grid search finished.")


[0 / 40]
[1 / 40]
[2 / 40]
[3 / 40]
[4 / 40]
[5 / 40]
[6 / 40]
[7 / 40]
[8 / 40]
[9 / 40]
[10 / 40]
[11 / 40]
[12 / 40]
[13 / 40]
[14 / 40]
[15 / 40]
[16 / 40]
[17 / 40]
[18 / 40]
[19 / 40]
[20 / 40]
[21 / 40]
[22 / 40]
[23 / 40]
[24 / 40]
[25 / 40]
[26 / 40]
[27 / 40]
[28 / 40]
[29 / 40]
[30 / 40]
[31 / 40]
[32 / 40]
[33 / 40]
[34 / 40]
[35 / 40]
[36 / 40]
[37 / 40]
[38 / 40]
[39 / 40]
Grid search finished.


## 6. Inspect the final in-memory model pool

In [22]:
def count_models(models_ram: dict) -> int:
    return sum(len(category_models) for category_models in models_ram.values())

def count_categories(models_ram: dict) -> int:
    return len(models_ram)

print(f"Categories kept : {count_categories(models_ram)}")
print(f"Models kept     : {count_models(models_ram)}")

print_models_ram_metrics(models_ram=models_ram)


Categories kept : 5
Models kept     : 15

MODELS RAM METRICS

--- CATEGORY: mcc ---
                       model_id category   thr     score       acc  precision  \
0  model-20260423_182243_039568      mcc  0.51  0.197763  0.598338   0.578378   
1  model-20260423_182243_faf6f0      mcc  0.48  0.166522  0.565097   0.532075   
2  model-20260423_182252_9d7528      mcc  0.51  0.166120  0.578947   0.552381   

     recall        f1       mcc  signals  signals_percent  abs_signals  \
0  0.614943  0.596100  0.197763      185         0.512465        0.185   
1  0.810345  0.642369  0.166522      265         0.734072        0.265   
2  0.666667  0.604167  0.166120      210         0.581717        0.210   

   abs_percent  
0        0.361  
1        0.361  
2        0.361  

--- CATEGORY: signals10 ---
                       model_id   category   thr     score       acc  \
0  model-20260423_182249_78fa23  signals10  0.55  0.673077  0.567867   
1  model-20260423_182250_23f067  signals10  0.55  0.6

## 7. What this demo notebook shows

This notebook demonstrates the main ML workflow of the demo project:

- downloading recent market data
- preprocessing ETH / BTC / ETHBTC candles
- loading train/test configs
- running grid search
- keeping the strongest candidate models in memory
- inspecting final predictions

The full private version of the project also includes:
- model persistence and loading
- model metadata analysis
- backtesting and grid-backtesting
- system-management helper pipelines
- ongoing work on step-by-step retraining and AI-agent integration
